# 03 — Dataset ponto × hora (dias 5–6)

Pré-requisito: `python -m src.dataset --versao v1`. Este notebook **valida** o dataset e gera os
números do `DATASET.md`.

In [ ]:
# Preparação: permite importar src/ a partir da pasta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 50)
print("Raiz do projeto:", RAIZ)

In [ ]:
from src.dataset import carregar

VERSAO = "v1"   # use "sintetico" para testar o notebook sem dados reais
dataset, resumo = carregar(VERSAO)
dataset.info()
import json
print(json.dumps(resumo["por_split"], indent=2, ensure_ascii=False))
print(json.dumps(resumo["exclusoes"], indent=2))

## Checagens de sanidade

In [ ]:
from src.features import COLUNAS_CHUVA

assert dataset[COLUNAS_CHUVA].isna().sum().sum() == 0, "chuva com NaN"
assert (dataset[COLUNAS_CHUVA] >= 0).all().all(), "chuva negativa"
assert (dataset["chuva_1h"] <= dataset["chuva_3h"] + 1e-3).all()
assert (dataset["chuva_24h"] <= dataset["chuva_72h"] + 1e-3).all()
assert dataset.loc[dataset.split == "treino", "ano"].max() <= config.ANO_FIM_TREINO
assert (dataset.loc[dataset.split != "treino", "peso_amostra"] == 1).all()
print("ok")

## Positivos × chuva
Os positivos devem se concentrar em horas com chuva recente. Se não, suspeite de fuso horário ou geocodificação.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for rotulo, g in dataset.groupby("alagou_3h"):
    np.log1p(g["chuva_3h"]).plot.hist(bins=40, alpha=0.5, density=True, ax=ax[0], label=f"alagou_3h={rotulo}")
ax[0].set_title("log(1 + chuva_3h)"); ax[0].legend()
dataset.groupby("hora_dia")["alagou_3h"].mean().plot.bar(ax=ax[1], title="Taxa de positivos por hora local")
plt.tight_layout()

In [ ]:
# Taxa de positivos por faixa de chuva nas últimas 3 h (ponderada pelo peso de amostragem)
faixas = pd.cut(dataset["chuva_3h"], [-0.1, 0.2, 2, 5, 10, 20, 40, 1000])
tab = dataset.assign(faixa=faixas).groupby("faixa", observed=True).apply(
    lambda g: pd.Series({"linhas": len(g), "positivos": g.alagou_3h.sum(),
                         "taxa_real": np.average(g.alagou_3h, weights=g.peso_amostra)}))
tab

In [ ]:
# Checagem de fuso: a chuva no momento dos eventos deveria ser maior que a média
pos = dataset[dataset.alagou_3h == 1]
print("chuva_1h média — positivos:", pos.chuva_1h.mean().round(2),
      "| todos (ponderado):", np.average(dataset.chuva_1h, weights=dataset.peso_amostra).round(3))

## Por zona e por ano

In [ ]:
dataset.pivot_table(index="ano", columns="zona", values="alagou_3h", aggfunc="sum", fill_value=0)

Copie as contagens para `DATASET.md` (seções 5 e 6).